## 1) Création de l'entité initiale

On commence avec un catalogue de fichiers CSV qui contiennent les données de nos entreprises. On cherche à trouver, stocker et analyser les données des trois entreprises : **Google** (`0878.065.378`), **Apple** (`0836.157.420`) et **SNCB** (`0203.430.576`).

Les fichiers KBO Open Data se trouvent dans `/home/jovyan/work/data/KBO/` :

| Fichier | Contenu |
|---|---|
| `enterprise.csv` | Statut, forme juridique, date de création |
| `denomination.csv` | Dénominations officielles (FR / NL) |
| `address.csv` | Adresses (siège, exploitation…) |
| `activity.csv` | Codes d'activité NACE |
| `contact.csv` | Téléphone, e-mail, site web |
| `establishment.csv` | Unités d'établissement |
| `code.csv` | Table de correspondance des codes → libellés |

In [55]:
import json
import re
import time
from collections import OrderedDict
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import requests
from bs4 import BeautifulSoup
from plotly.subplots import make_subplots

In [56]:

DATA_DIR = Path("data")  

GOOGLE_NUM = "0878.065.378"
APPLE_NUM  = "0836.157.420"
SNCB_NUM   = "0203.430.576"

FILES = ["enterprise", "denomination", "address", "activity",
         "contact", "establishment", "code"]

kbo = {name: pd.read_csv(DATA_DIR / f"{name}.csv", dtype=str) for name in FILES}

for name, df in kbo.items():
    print(f"{name:15s} {df.shape}")

enterprise      (1951671, 7)
denomination    (3346292, 4)
address         (2881017, 13)
activity        (34510460, 5)
contact         (706353, 4)
establishment   (1687516, 3)
code            (21468, 4)


In [57]:
def _entity_col(df):
    for c in ["EntityNumber", "EnterpriseNumber"]:
        if c in df.columns:
            return c
    return df.columns[0]

def get_entity(num):
    out = {"num": num}
    for name, df in kbo.items():
        if name == "code":
            continue
        col = _entity_col(df)
        out[name] = df[df[col] == num]
    return out

def _val(df, col):
    if len(df) and col in df.columns:
        v = df.iloc[0][col]
        return "" if pd.isna(v) else str(v)
    return ""

def _g(row, col):
    if col in row.index and not pd.isna(row[col]):
        return str(row[col])
    return ""

def _main_name(den):
    if len(den):
        official = den[den["TypeOfDenomination"] == "001"]
        src = official if len(official) else den
        return src.iloc[0]["Denomination"]
    return "(sans dénomination)"

def afficher_entity(e):
    ent  = e["enterprise"]
    den  = e["denomination"]
    addr = e["address"]
    act  = e["activity"]
    ct   = e["contact"]

    print("=" * 60)
    print(f"  {_main_name(den)}  ({e['num']})")
    print("=" * 60)
    print(f"  {'Statut':<19}: {_val(ent, 'Status')}")
    print(f"  {'Situation juridique':<19}: {_val(ent, 'JuridicalSituation')}")
    print(f"  {'Forme juridique':<19}: {_val(ent, 'JuridicalForm')}")
    print(f"  {"Type d\\'entreprise":<19}: {_val(ent, 'TypeOfEnterprise')}")
    print(f"  {'Date de création':<19}: {_val(ent, 'StartDate')}")

    address_str = ""
    if len(addr):
        a = addr.iloc[0]
        street = _g(a, "StreetFR") or _g(a, "StreetNL")
        house  = _g(a, "HouseNumber")
        box    = _g(a, "Box")
        zip_   = _g(a, "Zipcode")
        city   = _g(a, "MunicipalityFR") or _g(a, "MunicipalityNL")
        line = f"{street} {house}".strip()
        if box:
            line += f" bte {box}"
        address_str = f"{line}, {zip_} {city}".strip(", ").strip()
    print(f"  {'Adresse':<19}: {address_str}")

    print("  Activités")
    if len(act):
        for ver in sorted(act["NaceVersion"].unique(), reverse=True):
            print(f"    NACE {ver} :")
            for _, r in act[act["NaceVersion"] == ver].iterrows():
                print(f"      {_g(r,'NaceCode')}  [{_g(r,'Classification')}]  groupe {_g(r,'ActivityGroup')}")
    def _contacts(t):
        return list(ct[ct["ContactType"] == t]["Value"]) if len(ct) else []
    print(f"  {'Téléphone':<19}: {_contacts('TEL')}")
    print(f"  {'Email':<19}: {_contacts('EMAIL')}")
    print(f"  {'Site web':<19}: {_contacts('WEB')}")
    print(f"  {'Etablissements':<19}: {len(e['establishment'])}")
    print()
    print("  Dénominations :")
    if len(den):
        cols = ["Language", "TypeOfDenomination", "Denomination"]
        print(den[cols].to_string(index=False))
    else:
        print("  (aucune)")

### 🔍 Recherche — Google

In [58]:
google = get_entity(GOOGLE_NUM)
afficher_entity(google)

  GOOGLE BELGIUM  (0878.065.378)
  Statut             : AC
  Situation juridique: 000
  Forme juridique    : 014
  Type d\'entreprise : 2
  Date de création   : 21-12-2005
  Adresse            : Chaussée d'Etterbeek 180, 1040 Bruxelles
  Activités
    NACE 2025 :
      73110  [MAIN]  groupe 006
      62900  [MAIN]  groupe 001
    NACE 2008 :
      73110  [MAIN]  groupe 006
      62090  [MAIN]  groupe 001
    NACE 2003 :
      74401  [MAIN]  groupe 006
      72600  [MAIN]  groupe 001
  Téléphone          : []
  Email              : []
  Site web           : []
  Etablissements     : 1

  Dénominations :
Language TypeOfDenomination   Denomination
       2                001 GOOGLE BELGIUM


### 🔍 Recherche — Apple

In [59]:
apple = get_entity(APPLE_NUM)
afficher_entity(apple)

  APPLE RETAIL BELGIUM  (0836.157.420)
  Statut             : AC
  Situation juridique: 000
  Forme juridique    : 610
  Type d\'entreprise : 2
  Date de création   : 06-05-2011
  Adresse            : Boulevard Saint-Lazare 4-10, 1210 Saint-Josse-ten-Noode
  Activités
    NACE 2025 :
      47400  [MAIN]  groupe 006
      47400  [MAIN]  groupe 001
    NACE 2008 :
      47410  [MAIN]  groupe 006
      47410  [MAIN]  groupe 001
  Téléphone          : []
  Email              : []
  Site web           : []
  Etablissements     : 2

  Dénominations :
Language TypeOfDenomination         Denomination
       2                001 APPLE RETAIL BELGIUM


### 🔍 Recherche — SNCB

In [60]:
sncb = get_entity(SNCB_NUM)
afficher_entity(sncb)

  SOCIÉTÉ NATIONALE DES CHEMINS DE FER BELGES  (0203.430.576)
  Statut             : AC
  Situation juridique: 000
  Forme juridique    : 114
  Type d\'entreprise : 2
  Date de création   : 01-01-1968
  Adresse            : Rue de France 56, 1060 Saint-Gilles
  Activités
    NACE 2025 :
      80010  [SECO]  groupe 001
      49110  [MAIN]  groupe 001
    NACE 2008 :
      80100  [SECO]  groupe 001
      49100  [MAIN]  groupe 001
    NACE 2003 :
      74601  [SECO]  groupe 001
  Téléphone          : []
  Email              : []
  Site web           : []
  Etablissements     : 270

  Dénominations :
Language TypeOfDenomination                                    Denomination
       1                001     SOCIÉTÉ NATIONALE DES CHEMINS DE FER BELGES
       1                002                                            SNCB
       2                001 NATIONALE MAATSCHAPPIJ DER BELGISCHE SPOORWEGEN
       2                002                                            NMBS


### 🔄 Traduction des codes CSV

On traduit maintenant les codes issus des fichiers CSV en valeurs lisibles (libellés, catégories, etc.).

In [61]:
codes = kbo["code"]   

CODE_FR = {
    (r.Category, r.Code): r.Description
    for r in codes[codes["Language"] == "FR"].itertuples(index=False)
}

def traduire(category, code, defaut=""):
    if code is None or code == "" or (isinstance(code, float) and pd.isna(code)):
        return defaut
    return CODE_FR.get((category, str(code)), str(code))

print(traduire("Status", "AC"), "/", traduire("JuridicalForm", "610"),
      "/", traduire("TypeOfEnterprise", "2"))

Actif / Société à responsabilité limitée / Personne morale


In [62]:
def _entity_col(df):
    for c in ["EntityNumber", "EnterpriseNumber"]:
        if c in df.columns:
            return c
    return df.columns[0]

def get_entity(num):
    out = {"num": num}
    for name, df in kbo.items():
        if name == "code":
            continue
        out[name] = df[df[_entity_col(df)] == num]
    return out

def _val(df, col):
    if len(df) and col in df.columns:
        v = df.iloc[0][col]
        return "" if pd.isna(v) else str(v)
    return ""

def _g(row, col):
    return "" if (col not in row.index or pd.isna(row[col])) else str(row[col])

def _nom(e):
    den = e["denomination"]
    if not len(den):
        return "(sans dénomination)"
    soc = den[den["TypeOfDenomination"] == "001"]
    src = soc if len(soc) else den
    fr  = src[src["Language"] == "1"]
    return (fr if len(fr) else src).iloc[0]["Denomination"]

def _adresse(e):
    addr = e["address"]
    if not len(addr):
        return "", ""
    a = addr.iloc[0]
    street = _g(a, "StreetFR") or _g(a, "StreetNL")
    house, box = _g(a, "HouseNumber"), _g(a, "Box")
    zip_  = _g(a, "Zipcode")
    city  = _g(a, "MunicipalityFR") or _g(a, "MunicipalityNL")
    line  = f"{street} {house}".strip()
    if box:
        line += f" bte {box}"
    full  = f"{line}, {zip_} {city}".strip().strip(",").strip()
    return full, traduire("TypeOfAddress", _g(a, "TypeOfAddress"))

def _nace_ordonnes(e):
    act, lignes, codes_list = e["activity"], [], []
    if len(act):
        for ver in sorted(act["NaceVersion"].unique(), reverse=True):
            lignes.append(f"    NACE {ver} :")
            for r in act[act["NaceVersion"] == ver].itertuples(index=False):
                classif = traduire("Classification", getattr(r, "Classification", ""))
                nace    = traduire(f"Nace{ver}", r.NaceCode)
                groupe  = traduire("ActivityGroup", getattr(r, "ActivityGroup", ""))
                lignes.append(f"      [{classif}]  {r.NaceCode} — {nace}  ({groupe})")
                codes_list.append(r.NaceCode)
    return lignes, codes_list

def _contact(e, t, defaut="—"):
    ct = e["contact"]
    if len(ct):
        vals = list(ct[ct["ContactType"] == t]["Value"])
        if vals:
            return ", ".join(vals)
    return defaut

def _resume(e):
    ent = e["enterprise"]
    return {
        "Nom":                 _nom(e),
        "Numéro BCE":          e["num"],
        "Statut":              traduire("Status", _val(ent, "Status")),
        "Situation juridique": traduire("JuridicalSituation", _val(ent, "JuridicalSituation")),
        "Forme juridique":     traduire("JuridicalForm", _val(ent, "JuridicalForm")),
        "Type d'entreprise":   traduire("TypeOfEnterprise", _val(ent, "TypeOfEnterprise")),
        "Date de création":    _val(ent, "StartDate"),
        "Nb établissements":   len(e["establishment"]),
    }

W = 22
def afficher_entity(e):
    r = _resume(e)
    adresse, type_adr = _adresse(e)
    nace_lignes, _ = _nace_ordonnes(e)

    print("=" * 60)
    print(f"  {r['Nom']}  ({e['num']})")
    print("=" * 60)
    for label, value in [
        ("Nom", r["Nom"]), ("Numéro BCE", r["Numéro BCE"]),
        ("Statut", r["Statut"]), ("Situation juridique", r["Situation juridique"]),
        ("Forme juridique", r["Forme juridique"]),
        ("Type d'entreprise", r["Type d'entreprise"]),
        ("Date de création", r["Date de création"]),
        ("Adresse", adresse), ("Type d'adresse", type_adr),
    ]:
        print(f"  {label:<{W}}: {value}")

    print("Activités")
    for ligne in nace_lignes:
        print(ligne)

    for label, value in [
        ("Téléphone", _contact(e, "TEL")), ("Email", _contact(e, "EMAIL")),
        ("Site web", _contact(e, "WEB")), ("Nb établissements", r["Nb établissements"]),
    ]:
        print(f"  {label:<{W}}: {value}")

def table_comparaison(entities):
    ordre = ["Numéro BCE", "Statut", "Situation juridique", "Forme juridique",
             "Type d'entreprise", "Date de création", "Codes NACE", "Nb établissements"]
    data = {}
    for e in entities:
        r = _resume(e)
        _, codes_nace = _nace_ordonnes(e)
        col = {**r, "Codes NACE": ", ".join(codes_nace)}
        data[r["Nom"]] = pd.Series(col).reindex(ordre)
    df = pd.DataFrame(data)
    df.index.name = "Entreprise"
    return df

In [63]:
google = get_entity(GOOGLE_NUM)
apple  = get_entity(APPLE_NUM)
sncb   = get_entity(SNCB_NUM)

for e in [google, apple, sncb]:
    afficher_entity(e)
    print()

table_comparaison([google, apple, sncb])

  GOOGLE BELGIUM  (0878.065.378)
  Nom                   : GOOGLE BELGIUM
  Numéro BCE            : 0878.065.378
  Statut                : Actif
  Situation juridique   : Situation normale
  Forme juridique       : Société anonyme
  Type d'entreprise     : Personne morale
  Date de création      : 21-12-2005
  Adresse               : Chaussée d'Etterbeek 180, 1040 Bruxelles
  Type d'adresse        : Siège
Activités
    NACE 2025 :
      [Activité principale]  73110 — Activités d’agence de publicité  (Activités ONSS)
      [Activité principale]  62900 — Autres activités de service informatique  (Activités TVA)
    NACE 2008 :
      [Activité principale]  73110 — Activités des agences de publicité  (Activités ONSS)
      [Activité principale]  62090 — Autres activités informatiques  (Activités TVA)
    NACE 2003 :
      [Activité principale]  74401 — Agences de publicité  (Activités ONSS)
      [Activité principale]  72600 — Autres activités rattachées à l'informatique  (Activités TVA)
 

,GOOGLE BELGIUM,APPLE RETAIL BELGIUM,SOCIÉTÉ NATIONALE DES CHEMINS DE FER BELGES
Entreprise,,,
Numéro BCE,0878.065.378,0836.157.420,0203.430.576
Statut,Actif,Actif,Actif
Situation juridique,Situation normale,Situation normale,Situation normale
Forme juridique,Société anonyme,Société à responsabilité limitée,Société anonyme de droit public
Type d'entreprise,Personne morale,Personne morale,Personne morale
Date de création,21-12-2005,06-05-2011,01-01-1968
Codes NACE,"73110, 62900, 73110, 62090, 74401, 72600","47400, 47400, 47410, 47410","80010, 49110, 80100, 49100, 74601"
Nb établissements,1,2,270


In [64]:
## output after code transformations

---
## 2) Informations supplémentaires externes (KBO)

À partir d'ici, on récupère des données plus riches depuis le **Carrefour des Entreprises Belges (KBO/BCE)**.

URL de référence :  
`https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer=<NUMERO>`

Exemple : https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer=203430576

In [65]:
GOOGLE_NUM_URL = "878065378"
APPLE_NUM_URL  = "836157420"
SNCB_NUM_URL   = "203430576"
BASE_URL = "https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; KBO-notebook/1.0)"}

def kbo_soup(num_url, lang="fr"):
    params = {"lang": lang, "ondernemingsnummer": num_url}
    r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=20)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")


---
### 2.1) Informations Générales

| Champ | Valeur |
|---|---|
| Dénomination | |
| Numéro d'entreprise | |
| Adresse | |
| Activité principale | |
| Effectif | |
| Date de création | |

In [66]:
def _kv(soup):
    kv = {}
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td", recursive=False)
        if len(tds) >= 2:
            label = tds[0].get_text(" ", strip=True).rstrip(":").strip()
            if label and label not in kv:
                kv[label] = tds[1].get_text("\n", strip=True)
    return kv

def _premiere_ligne(txt):
    return txt.split("\n")[0].strip() if txt else ""

def _adresse(txt):
    if not txt:
        return ""
    lignes = [l.strip() for l in txt.split("\n")
              if l.strip() and not l.strip().lower().startswith("depuis")]
    return ", ".join(lignes)

def _activite_principale(soup):
    flat = re.sub(r"\s+", " ", soup.get_text(" "))
    for v in ("2025", "2008", "2003"):
        m = re.search(rf"{v}\s+(\d{{2}}\.\d{{3}})\s*-\s*(.+?)\s+Depuis", flat)
        if m:
            return f"{m.group(1)} — {m.group(2).strip()} (NACE {v})"
    return ""

def infos_kbo_21(num_url):
    soup = kbo_soup(num_url)
    kv = _kv(soup)
    return {
        "Dénomination":        _premiere_ligne(kv.get("Dénomination", "")),
        "Numéro d'entreprise": kv.get("Numéro d'entreprise", ""),
        "Adresse":             _adresse(kv.get("Adresse du siège", "")),
        "Activité principale": _activite_principale(soup),
        "Effectif":            "Non publié dans la BCE Public Search",
        "Date de création":    kv.get("Date de début", ""),
    }

def afficher_kbo_21(num_url):
    infos = infos_kbo_21(num_url)
    nom, num = infos["Dénomination"], infos["Numéro d'entreprise"]
    print("=" * 60)
    print(f"  {nom}  ({num})")
    print("=" * 60)
    for k, v in infos.items():
        print(f"  {k:<20}: {v}")
    print()
    return infos

In [67]:
for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_kbo_21(n)
    time.sleep(1) 

  GOOGLE BELGIUM  (0878.065.378)
  Dénomination        : GOOGLE BELGIUM
  Numéro d'entreprise : 0878.065.378
  Adresse             : Chaussée d'Etterbeek 180, 1040 Bruxelles, Info supplémentaires:, GooglePlex
  Activité principale : 62.900 — Autres activités de service informatique (NACE 2025)
  Effectif            : Non publié dans la BCE Public Search
  Date de création    : 21 décembre 2005

  APPLE RETAIL BELGIUM  (0836.157.420)
  Dénomination        : APPLE RETAIL BELGIUM
  Numéro d'entreprise : 0836.157.420
  Adresse             : Boulevard Saint-Lazare 4-10, 1210 Saint-Josse-ten-Noode, Info supplémentaires:, Botanic Tower 6de verdieping
  Activité principale : 47.400 — Commerce de détail d’équipements de l’information et de la communication (NACE 2025)
  Effectif            : Non publié dans la BCE Public Search
  Date de création    : 6 mai 2011

  SOCIÉTÉ NATIONALE DES CHEMINS DE FER BELGES  (0203.430.576)
  Dénomination        : SOCIÉTÉ NATIONALE DES CHEMINS DE FER BELGES
  N

---
### 2.2) Informations Juridiques

| Champ | Valeur |
|---|---|
| Forme juridique | |
| Numéro de TVA | |
| Situation juridique | |
| Capital social | |
| Assemblée générale | |
| Date de fin de l'année comptable | |

In [68]:
def _sans_depuis(txt):
    if not txt:
        return ""
    t = txt.split("\n")[0]
    t = re.split(r"\s+Depuis\b", t)[0]
    return t.strip()

def _numero_tva(soup, kv):
    flat = re.sub(r"\s+", " ", soup.get_text(" "))
    num = re.sub(r"\D", "", kv.get("Numéro d'entreprise", ""))   # ex: 0203430576
    if "Assujettie à la TVA" in flat and num:
        return "BE" + num
    return "Non assujettie à la TVA"

def infos_kbo_22(num_url):
    soup = kbo_soup(num_url)
    kv = _kv(soup)
    return {
        "Forme juridique":                  _sans_depuis(kv.get("Forme légale", "")),
        "Numéro de TVA":                    _numero_tva(soup, kv),
        "Situation juridique":              _sans_depuis(kv.get("Situation juridique", "")),
        "Capital social":                   kv.get("Capital", "Pas de capital / non communiqué"),
        "Assemblée générale":               kv.get("Assemblée générale", "—"),
        "Date de fin de l'année comptable": kv.get("Date de fin de l'année comptable", "—"),
    }

def afficher_kbo_22(num_url):
    infos = infos_kbo_22(num_url)
    print("=" * 60)
    print(f"  Informations juridiques  ({num_url})")
    print("=" * 60)
    for k, v in infos.items():
        print(f"  {k:<34}: {v}")
    print()
    return infos

In [69]:
for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_kbo_22(n)
    time.sleep(1)

  Informations juridiques  (878065378)
  Forme juridique                   : Société anonyme
  Numéro de TVA                     : BE0878065378
  Situation juridique               : Situation normale
  Capital social                    : 770.000,00 EUR
  Assemblée générale                : juin
  Date de fin de l'année comptable  : 31 décembre

  Informations juridiques  (836157420)
  Forme juridique                   : Société à responsabilité limitée
  Numéro de TVA                     : BE0836157420
  Situation juridique               : Situation normale
  Capital social                    : Pas de capital / non communiqué
  Assemblée générale                : mars
  Date de fin de l'année comptable  : 24 septembre

  Informations juridiques  (203430576)
  Forme juridique                   : Société anonyme de droit public
  Numéro de TVA                     : BE0203430576
  Situation juridique               : Situation normale
  Capital social                    : 249.022.345,57 EU

---
### 2.3) Activités

Liste de chaque domaine d'activité avec son code NACE/ONSS.

| Code | Description |
|---|---|

In [70]:
def activites_kbo(num_url):
    soup = kbo_soup(num_url)
    flat = re.sub(r"\s+", " ", soup.get_text(" "))
    pat = re.compile(r"(TVA|ONSS|EMPL)\s*(2025|2008|2003)\s+(\d{2}\.\d{3})\s*-\s*(.+?)\s+Depuis")
    out = []
    for typ, ver, code, desc in pat.findall(flat):
        out.append({
            "code": code.replace(".", ""),                      
            "description": f"{desc.strip()} (NACE {ver} · {typ})"
        })
    return out

In [71]:
def afficher_activites(num_url):
    acts = activites_kbo(num_url)
    df = pd.DataFrame(acts, columns=["code", "description"])
    df.columns = ["Code", "Description"]
    print("=" * 60)
    print(f"  Activités — {num_url}  ({len(df)} ligne·s)")
    print("=" * 60)
    display(df)
    return acts

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_activites(n)

  Activités — 878065378  (6 ligne·s)


,Code,Description
0,62900,Autres activités de service informatique (NACE...
1,73110,Activités d’agence de publicité (NACE 2025 · O...
2,62090,Autres activités informatiques (NACE 2008 · TVA)
3,73110,Activités des agences de publicité (NACE 2008 ...
4,72600,Autres activités rattachées à l'informatique (...
5,74401,Agences de publicité (NACE 2003 · ONSS)


  Activités — 836157420  (4 ligne·s)


,Code,Description
0,47400,Commerce de détail d’équipements de l’informat...
1,47400,Commerce de détail d’équipements de l’informat...
2,47410,"Commerce de détail d'ordinateurs, d'unités pér..."
3,47410,"Commerce de détail d'ordinateurs, d'unités pér..."


  Activités — 203430576  (5 ligne·s)


,Code,Description
0,80010,Activités d’investigation et de sécurité privé...
1,49110,Transport ferroviaire lourd de voyageurs (NACE...
2,80100,Activités de sécurité privée (NACE 2008 · TVA)
3,49100,Transport ferroviaire de voyageurs autre qu'ur...
4,74601,Entreprise de gardiennage et service de sécuri...


---
### 2.4) Dirigeants et Représentants

Liste de chaque dirigeant avec sa ou ses qualité(s).

| Nom | Qualité(s) |
|---|---|

In [72]:
def dirigeants_kbo(num_url):
    soup = kbo_soup(num_url)
    pers = OrderedDict()
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td", recursive=False)
        if len(tds) != 3:
            continue
        fonction = tds[0].get_text(" ", strip=True)
        nom      = tds[1].get_text(" ", strip=True)
        depuis   = tds[2].get_text(" ", strip=True)
        if "," not in nom:
            continue
        if re.search(r"\d{2}\.\d{3}", fonction + nom):     
            continue
        if not depuis.lower().startswith("depuis"):
            continue
        nom = re.sub(r"\s*,\s*", ", ", nom).strip()
        pers.setdefault(nom, [])
        if fonction and fonction not in pers[nom]:
            pers[nom].append(fonction)
    return [{"nom": n, "qualites": q} for n, q in pers.items()]

In [73]:
def afficher_dirigeants(num_url):
    dirs = dirigeants_kbo(num_url)
    df = pd.DataFrame([{"Nom": d["nom"], "Qualité(s)": ", ".join(d["qualites"])}
                       for d in dirs])
    print("=" * 60)
    print(f"  Dirigeants & Représentants — {num_url}  ({len(df)} personne·s)")
    print("=" * 60)
    display(df if len(df) else "Aucun dirigeant repris dans la BCE.")
    return dirs

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_dirigeants(n)
    time.sleep(1)

  Dirigeants & Représentants — 878065378  (3 personne·s)


,Nom,Qualité(s)
0,"Fagan, Frank",Administrateur
1,"KARAIVANOV, SVILEN",Administrateur
2,"Turschl, Gabor","Administrateur, Administrateur délégué"


  Dirigeants & Représentants — 836157420  (2 personne·s)


,Nom,Qualité(s)
0,"GALA, TEJAS",Administrateur
1,"KAM, WAN KEI",Administrateur


  Dirigeants & Représentants — 203430576  (14 personne·s)


,Nom,Qualité(s)
0,"Boelaert, Filip",Administrateur
1,"Bonaventure, Chanelle",Administrateur
2,"Douette, Emmanuel",Administrateur
3,"Durez, Martine",Administrateur
4,"Georgin, Thibaut",Administrateur
5,"Glautier, Laurence",Administrateur
6,"Henry, Olivier",Administrateur
7,"Mercenier, Eric",Administrateur
8,"Poot, An",Administrateur
9,"Schalck, Daan",Administrateur


---
### 2.5) Liens entre Entités

Liste de chaque entité liée à l'entreprise.

| Entité | Numéro d'entreprise | Date du lien | Nature du lien | Statut actuel |
|---|---|---|---|---|

In [74]:
def liens_kbo(num_url, statut=True, pause=0.5):
    soup = kbo_soup(num_url)
    liens, seen = [], set()
    for a in soup.find_all("a", href=True):
        if "toonondernemingps.html" not in a["href"] or "ondernemingsnummer=" not in a["href"]:
            continue
        numero = a.get_text(" ", strip=True)
        if not re.match(r"\d{4}\.\d{3}\.\d{3}", numero):
            continue
        container, node = None, a
        for _ in range(5):
            node = node.parent
            if node is None:
                break
            if "depuis le" in node.get_text(" ", strip=True).lower():
                container = node
                break
        txt = container.get_text(" ", strip=True) if container else numero
        m_name = re.search(r"\((.+?)\)", txt)
        entite = m_name.group(1).strip() if m_name else ""
        after  = txt.split(")", 1)[1] if ")" in txt else txt
        m = re.search(r"(.*?)\s*depuis le\s*(.+)$", after, re.I)
        nature    = (m.group(1) if m else after).strip()
        date_lien = m.group(2).strip() if m else ""

        key = (numero, nature, date_lien)
        if key in seen:
            continue
        seen.add(key)
        liens.append({"entite": entite, "numero": numero,
                      "date_lien": date_lien, "nature": nature, "statut": "—"})

    if statut:                      
        for l in liens:
            try:
                s = kbo_soup(re.sub(r"\D", "", l["numero"]).lstrip("0"))
                l["statut"] = _kv(s).get("Statut", "—")
            except Exception:
                l["statut"] = "Indisponible"
            time.sleep(pause)
    return liens

In [75]:
def afficher_liens(num_url, statut=True):
    liens = liens_kbo(num_url, statut=statut)
    df = pd.DataFrame([{
        "Entité": l["entite"],
        "Numéro d'entreprise": l["numero"],
        "Date du lien": l["date_lien"],
        "Nature du lien": l["nature"],
        "Statut actuel": l["statut"],
    } for l in liens])
    print("=" * 60)
    print(f"  Liens entre entités — {num_url}  ({len(df)} lien·s)")
    print("=" * 60)
    display(df if len(df) else "Aucun lien repris dans la BCE.")
    return liens

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_liens(n, statut=True)
    time.sleep(1)

  Liens entre entités — 878065378  (0 lien·s)


'Aucun lien repris dans la BCE.'

  Liens entre entités — 836157420  (0 lien·s)


'Aucun lien repris dans la BCE.'

  Liens entre entités — 203430576  (13 lien·s)


,Entité,Numéro d'entreprise,Date du lien,Nature du lien,Statut actuel
0,Financière TGV - HST-FIN,0260.581.887,10 septembre 2004,est absorbée par cette entité,Arrêté\nDepuis le 10 septembre 2004
1,L.A. GROUP,0419.345.054,29 juin 2012,est absorbée par cette entité,Arrêté\nDepuis le 29 juin 2012
2,Société nationale des Chemins de Fer belges,0869.763.069,1 janvier 2014,est absorbée par cette entité,Arrêté\nDepuis le 1 janvier 2014
3,FONCIERE RUE DE FRANCE - VASTGOEDMAATSCHAPPIJ ...,0433.939.101,1 janvier 2016,est absorbée par cette entité,Arrêté\nDepuis le 1 janvier 2016
4,SOUTH STATION,0896.513.095,1 janvier 2016,est absorbée par cette entité,Arrêté\nDepuis le 1 janvier 2016
5,SPV LLN,0826.478.107,22 décembre 2017,est absorbée par cette entité,Arrêté\nDepuis le 22 décembre 2017
6,EUROSTATION,0446.601.757,26 octobre 2018,est absorbée par cette entité,Arrêté\nDepuis le 26 octobre 2018
7,B-PARKING,0899.438.834,2 juillet 2021,est absorbée par cette entité,Arrêté\nDepuis le 2 juillet 2021
8,Eurogare,0451.150.562,17 décembre 2021,est absorbée par cette entité,Arrêté\nDepuis le 17 décembre 2021
9,Railtour,0402.698.765,25 février 2022,est absorbée par cette entité,Arrêté\nDepuis le 25 février 2022


---
### 2.6) Documents Juridiques (Statuts)

Source : https://statuts.notaire.be/stapor_v1/enterprise/{numero}/statutes

| Document | Date | Lien |
|---|---|---|

In [76]:
def statuts_kbo(num_url, limit=5, max_pages=50, cookie=None, debug=False):
    numero = num_url.zfill(10)
    url = f"https://statuts.notaire.be/stapor_v1/api/enterprises/{numero}/statutes"
    headers = {
        **HEADERS,
        "Accept": "application/json, text/plain, */*",
        "Referer": f"https://statuts.notaire.be/stapor_v1/enterprise/{numero}/statutes",
        "X-Requested-With": "XMLHttpRequest",
    }
    if cookie:
        headers["Cookie"] = cookie

    items, offset, total = [], 0, None
    for _ in range(max_pages):
        params = {"deedDate": "", "offset": offset, "limit": limit}
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
        except Exception as e:
            if debug: print("  erreur réseau:", e)
            break
        ctype = r.headers.get("content-type", "")
        if debug:
            print(f"offset={offset} status={r.status_code} len={len(r.text)} ctype={ctype}")
        if r.status_code == 429:
            print("  ⚠️ 429 Too Many Requests — attends 2-3 min."); break
        if r.status_code != 200 or "json" not in ctype.lower():
            if debug: print("  -> pas de JSON (mur anti-bot ou cookie manquant/expiré)")
            break
        data  = r.json()
        items.extend(data.get("statutes", []))
        total = data.get("totalItems", total)
        offset += limit
        if (total is not None and offset >= total):
            break
    return items                    

In [77]:
import json
from pathlib import Path

COOKIE_FILE = Path("notaire_cookies.json")   # cache disque (comme le prof)
SEED_BCE    = "0836157420"                   # Apple, pour le challenge initial

def _cookies_to_str(cookies):
    """Liste de cookies Playwright -> string 'a=1; b=2' pour l'en-tête Cookie."""
    return "; ".join(f"{c['name']}={c['value']}" for c in cookies)

def _cookie_valide(cookie_str):
    """Ping rapide : True si l'API renvoie du JSON avec ce cookie."""
    if not cookie_str:
        return False
    try:
        r = requests.get(
            f"https://statuts.notaire.be/stapor_v1/api/enterprises/{SEED_BCE}/statutes",
            params={"offset": 0, "limit": 1},
            headers={**HEADERS, "Accept": "application/json", "Cookie": cookie_str},
            timeout=10,
        )
        return "application/json" in r.headers.get("content-type", "")
    except Exception:
        return False

import subprocess, sys, json

_PW_CHILD = r'''
import sys, json
from playwright.sync_api import sync_playwright

SEED_BCE = "0836157420"
seed = (f"https://statuts.notaire.be/stapor_v1/enterprise/{SEED_BCE}/statutes"
        f"?enterpriseNumber={SEED_BCE}&statuteStart=0&statuteCount=5")

with sync_playwright() as p:
    try:
        browser = p.chromium.launch(channel="chrome", headless=False)
    except Exception:
        browser = p.chromium.launch(headless=False)
    ctx  = browser.new_context(locale="fr-BE")
    page = ctx.new_page()
    page.add_init_script("Object.defineProperty(navigator,'webdriver',{get:()=>undefined})")
    page.goto("https://statuts.notaire.be/", wait_until="load", timeout=20000)
    page.wait_for_timeout(2000)
    page.goto(seed, wait_until="load", timeout=30000)
    for _ in range(40):
        names = {c["name"] for c in ctx.cookies()}
        if "Lyp1CWKh" in names and ("OCImoOot" in names or "OClmoOot" in names):
            break
        page.wait_for_timeout(500)
    cookies = ctx.cookies()
    browser.close()

sys.stdout.write("__COOKIES__" + json.dumps(cookies))
'''

def _renouveler_cookie_playwright():
    """Lance Playwright dans un sous-processus (évite le conflit asyncio de Jupyter)."""
    res = subprocess.run([sys.executable, "-c", _PW_CHILD],
                         capture_output=True, text=True, encoding="utf-8", timeout=120)
    out = res.stdout or ""
    if "__COOKIES__" not in out:
        print("⚠️ Sortie inattendue du sous-processus :\n", out[-500:],
              "\n--- stderr ---\n", (res.stderr or "")[-800:])
        return []
    return json.loads(out.split("__COOKIES__", 1)[1])

def get_cookie_notaire(force=False):
    """Renvoie une string cookie valide. Cache disque + renouvellement auto Playwright."""
    if not force and COOKIE_FILE.exists():
        cookie_str = _cookies_to_str(json.loads(COOKIE_FILE.read_text()))
        if _cookie_valide(cookie_str):
            print("Cookie notaire OK (cache)")
            return cookie_str
        print("Cookie expiré — renouvellement…")
    cookies = _renouveler_cookie_playwright()
    COOKIE_FILE.write_text(json.dumps(cookies, indent=2))
    print(f"Cookie renouvelé → {COOKIE_FILE}")
    return _cookies_to_str(cookies)

In [78]:
COOKIE_NOTAIRE = get_cookie_notaire()       

brut = statuts_kbo(GOOGLE_NUM_URL, cookie=COOKIE_NOTAIRE, debug=True)
print("\nTotal récupéré :", len(brut))
if brut:
    print(json.dumps(brut[0], indent=2, ensure_ascii=False))

Cookie notaire OK (cache)
offset=0 status=200 len=426 ctype=application/json

Total récupéré : 1
{
  "statutesHistoryId": 291205,
  "documentId": 512412,
  "documentStatus": "DONE",
  "documentTitle": "GOOGLE BELGIUM",
  "lastModificationDate": "2022-07-08T11:46:01.51Z",
  "deedDate": "2022-07-07",
  "organizationName": "NOTARIS STIJN RAES",
  "organizationId": "110932"
}


In [79]:
def _premier_lien(item):
    trouves = []
    def walk(o):
        if isinstance(o, dict):
            for v in o.values(): walk(v)
        elif isinstance(o, list):
            for x in o: walk(x)
        elif isinstance(o, str) and (o.startswith("http") or o.lower().endswith(".pdf")):
            trouves.append(o)
    walk(item)
    return trouves[0] if trouves else ""

def _pick(cols, *needles):
    for n in needles:
        for low, orig in cols.items():
            if n in low:
                return orig
    return None

DOC_URL = "https://statuts.notaire.be/stapor_v1/api/documents/{}" 

def afficher_statuts(num_url, cookie=None):
    items = statuts_kbo(num_url, cookie=cookie)
    print("=" * 60)
    print(f"  Documents juridiques — {num_url}  ({len(items)} doc·s)")
    print("=" * 60)
    if not items:
        print("  Aucun statut disponible "
              "(notaire.be ne reprend que les actes transmis depuis le 01/05/2019).\n")
        return []
    rows = [{
        "Document": s.get("documentTitle", ""),
        "Date":     s.get("deedDate", ""),
        "Étude":    s.get("organizationName", ""),
        "Lien":     DOC_URL.format(s["documentId"]) if s.get("documentId") else "",
    } for s in items]
    df = pd.DataFrame(rows, columns=["Document", "Date", "Étude", "Lien"])
    display(df)
    return items

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_statuts(n, cookie=COOKIE_NOTAIRE)
    time.sleep(1)

  Documents juridiques — 878065378  (1 doc·s)


,Document,Date,Étude,Lien
0,GOOGLE BELGIUM,2022-07-07,NOTARIS STIJN RAES,https://statuts.notaire.be/stapor_v1/api/docum...


  Documents juridiques — 836157420  (1 doc·s)


,Document,Date,Étude,Lien
0,APPLE RETAIL BELGIUM - statuten op 31.08.2021,2021-08-31,"VRONINKS, RICKER & WEYTS & SACRÉ - notaires",https://statuts.notaire.be/stapor_v1/api/docum...


  Documents juridiques — 203430576  (0 doc·s)
  Aucun statut disponible (notaire.be ne reprend que les actes transmis depuis le 01/05/2019).



---
### 2.7) Comptes Annuels

Source : https://consult.cbso.nbb.be/consult-enterprise/{numero}

**Stratégie de stockage :**
- **PDFs** — tous les dépôts depuis l'an 2000 → stockés sur HDFS sous `/data/nbb/pdfs/{numero}/{annee}.pdf`
- **CSVs** — tous les dépôts depuis 2021 (format tabulaire) → stockés sur HDFS sous `/data/nbb/csvs/{numero}/{annee}.csv`

> Les dépôts antérieurs à 2021 n'ont pas de CSV disponible (fichiers migrés — PDF uniquement).
> Exclure les comptes **consolidés** (modèle `mc-*`) et dédupliquer à **un seul dépôt par année** (préférer FR sur NL).

| Année fiscale | Date de publication | PDF | CSV |
|---|---|---|---|

In [80]:
CBSO_API     = "https://consult.cbso.nbb.be/api/rs-consult/published-deposits"
CBSO_HEADERS = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}
NBB_BASE     = Path("data/nbb")   

def cbso_deposits(numero, size=100, debug=False):
    numero = numero.replace(".", "").zfill(10)
    items, page = [], 0
    while True:
        params = {"page": page, "size": size, "enterpriseNumber": numero,
                  "sort": ["periodEndDate,desc", "depositDate,desc"]}
        r = requests.get(CBSO_API, headers=CBSO_HEADERS, params=params, timeout=30)
        if r.status_code != 200:
            if debug: print("HTTP", r.status_code, r.text[:200])
            break
        data  = r.json()
        batch = data.get("content", [])
        items.extend(batch)
        if debug and page == 0:
            print("totalPages:", data.get("totalPages"), "| totalElements:", data.get("totalElements"))
        if data.get("last", True) or not batch:
            break
        page += 1
        time.sleep(0.3)
    return items

In [81]:
raw = cbso_deposits("0203430576", debug=True)   
print("Dépôts récupérés :", len(raw))
print(json.dumps(raw[0], indent=2, ensure_ascii=False))

totalPages: 2 | totalElements: 153
Dépôts récupérés : 153
{
  "periodEndDateYear": 2025,
  "creationDate": "2026-06-16T01:20:39.290Z",
  "currency": {
    "code": "EUR",
    "name": "Euro",
    "legacy": false
  },
  "depositDate": "2026-06-09T08:41:53.210Z",
  "enterpriseNumber": "0203430576",
  "id": "b69da26d-63cf-11f1-9e09-db0606b28a82",
  "language": "FR",
  "modelId": "m120-f-p",
  "modelName": "Modèle pour comptes consolidés",
  "modificationDate": "2026-06-16T01:20:39.290Z",
  "periodEndDate": "2025-12-31T00:00:00Z",
  "periodStartDate": "2025-01-01T00:00:00Z",
  "status": "PUBLISHED",
  "taxonomyName": "Modèles 2021 (23.0.9)",
  "type": "Initial",
  "subType": null,
  "reference": "2026-00162246",
  "importFileType": "PDF",
  "generalAssemblyApprovalDate": "2026-05-29T00:00:00Z",
  "enterpriseName": "SOCIÉTÉ NATIONALE DES CHEMINS DE FER BELGES",
  "migration": false,
  "csrdFileType": null,
  "cbcrFileType": null
}


In [82]:
GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL = "0878065378", "0836157420", "0203430576"

def _year(dep):
    y = dep.get("periodEndDateYear") or str(dep.get("periodEndDate", ""))[:4]
    return int(y) if str(y).isdigit() else None

def _is_fr(dep):
    return (dep.get("language") or "").upper() == "FR"

def _is_consolidated(dep):
    mid  = (dep.get("modelId") or "").lower()
    name = (dep.get("modelName") or "").lower()
    return mid.startswith(("m120", "m122", "mc")) or "consolid" in name or "geconsolideerde" in name


def comptes_retenus(numero, an_min=2021, an_max=2025):
    numero = numero.replace(".", "").zfill(10)
    par_an = {}
    for d in cbso_deposits(numero):
        if _is_consolidated(d):                       
            continue
        y = d.get("periodEndDateYear")
        if not (isinstance(y, int) and an_min <= y <= an_max):   
            continue
        cur = par_an.get(y)
        if cur is None:
            par_an[y] = d
        else:
            cand_better = (_is_fr(d) and not _is_fr(cur))
            cur_is_p    = (cur.get("modelId","").endswith("-p"))
            cand_is_p   = (d.get("modelId","").endswith("-p"))
            if cand_better or (cur_is_p and not cand_is_p):
                par_an[y] = d
    return dict(sorted(par_an.items(), reverse=True))

def table_comptes(numero):
    retenus = comptes_retenus(numero)
    rows = [{
        "Année fiscale":       y,
        "Date de publication": d.get("depositDate"),
        "PDF":  "✓" if y >= 2000 else "—",          
        "CSV":  "✓" if y >= 2021 else "—",         
        "Langue":    "FR" if _is_fr(d) else "NL",
        "Référence": d.get("reference"),
    } for y, d in retenus.items()]
    df = pd.DataFrame(rows)
    print("=" * 60)
    print(f"  Comptes annuels — {numero}  ({len(df)} année·s retenue·s)")
    print("=" * 60)
    display(df)
    return retenus

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    table_comptes(n)

  Comptes annuels — 0878065378  (5 année·s retenue·s)


,Année fiscale,Date de publication,PDF,CSV,Langue,Référence
0,2025,2026-06-10T13:36:42.857Z,✓,✓,NL,2026-00149705
1,2024,2025-06-25T12:39:00.260Z,✓,✓,NL,2025-00184871
2,2023,2024-07-05T08:59:35.277Z,✓,✓,NL,2024-00218430
3,2022,2023-07-10T12:21:25.523Z,✓,✓,NL,2023-00223819
4,2021,2022-07-06T09:26:34.363Z,✓,✓,NL,2022-20180306


  Comptes annuels — 0836157420  (5 année·s retenue·s)


,Année fiscale,Date de publication,PDF,CSV,Langue,Référence
0,2025,2026-03-17T17:03:00.297Z,✓,✓,NL,2026-00061904
1,2024,2025-04-01T15:42:42.767Z,✓,✓,NL,2025-00064730
2,2023,2024-03-21T15:21:19.880Z,✓,✓,NL,2024-00051713
3,2022,2023-03-22T15:50:00.370Z,✓,✓,NL,2023-00044760
4,2021,2022-03-15T00:00:00Z,✓,✓,NL,2022-08100028


  Comptes annuels — 0203430576  (5 année·s retenue·s)


,Année fiscale,Date de publication,PDF,CSV,Langue,Référence
0,2025,2026-06-08T09:56:50.793Z,✓,✓,FR,2026-00162824
1,2024,2025-06-04T08:33:51.197Z,✓,✓,FR,2025-00124140
2,2023,2024-06-06T07:02:39.727Z,✓,✓,FR,2024-00119626
3,2022,2023-06-05T10:27:34.253Z,✓,✓,FR,2023-00112515
4,2021,2022-06-01T08:39:37.817Z,✓,✓,FR,2022-20049631


In [83]:
BROKER = "https://consult.cbso.nbb.be/api/external/broker/public/deposits"

def download_file(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, headers={
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*",
        "Referer": "https://consult.cbso.nbb.be/",
    }, timeout=120)
    if r.status_code == 200 and r.content:
        path.write_bytes(r.content)
        return True
    print("  échec", r.status_code, url)
    return False

def download_comptes(numero):
    numero = numero.replace(".", "").zfill(10)
    for annee, dep in comptes_retenus(numero).items():
        did = dep["id"]
        ok_pdf = download_file(f"{BROKER}/pdf/{did}",          
                               NBB_BASE/"pdfs"/numero/f"{annee}.pdf")
        ok_csv = False
        if annee >= 2021:                                  
            ok_csv = download_file(f"{BROKER}/consult/csv/{did}",
                                   NBB_BASE/"csvs"/numero/f"{annee}.csv")
        print(numero, annee, dep.get("reference"),
              "PDF ✓" if ok_pdf else "PDF —",
              "CSV ✓" if ok_csv else ("CSV —" if annee >= 2021 else "CSV n/a"))
        time.sleep(0.4)

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    download_comptes(n)

0878065378 2025 2026-00149705 PDF ✓ CSV ✓
0878065378 2024 2025-00184871 PDF ✓ CSV ✓
0878065378 2023 2024-00218430 PDF ✓ CSV ✓
0878065378 2022 2023-00223819 PDF ✓ CSV ✓
0878065378 2021 2022-20180306 PDF ✓ CSV ✓
0836157420 2025 2026-00061904 PDF ✓ CSV ✓
0836157420 2024 2025-00064730 PDF ✓ CSV ✓
0836157420 2023 2024-00051713 PDF ✓ CSV ✓
0836157420 2022 2023-00044760 PDF ✓ CSV ✓
  échec 500 https://consult.cbso.nbb.be/api/external/broker/public/deposits/consult/csv/B442B163-A126-4EF3-9080-75378F781185
0836157420 2021 2022-08100028 PDF ✓ CSV —
0203430576 2025 2026-00162824 PDF ✓ CSV ✓
0203430576 2024 2025-00124140 PDF ✓ CSV ✓
0203430576 2023 2024-00119626 PDF ✓ CSV ✓
0203430576 2022 2023-00112515 PDF ✓ CSV ✓
0203430576 2021 2022-20049631 PDF ✓ CSV ✓


In [84]:
numero = SNCB_NUM_URL.replace(".", "").zfill(10)

deps = cbso_deposits(numero)
print("Dépôts bruts renvoyés par l'API :", len(deps))

from collections import Counter
print("\nModèles présents (modelId / modelName) :")
for (mid, mname), n in Counter((d.get("modelId"), d.get("modelName")) for d in deps).most_common():
    print(f"  {str(mid):10s} {n:3d}×  {mname}")

print("\nAprès filtres (comptes_retenus) :", len(comptes_retenus(numero)), "année(s)")

if deps:
    import json
    print("\nExemple de dépôt :")
    print(json.dumps({k: deps[0][k] for k in ("periodEndDateYear","modelId","modelName","language","reference","id")}, indent=2, ensure_ascii=False))

Dépôts bruts renvoyés par l'API : 100

Modèles présents (modelId / modelName) :
  m120-f-p    16×  Model voor geconsolideerde jaarrekening
  m120-f-p    14×  Modèle pour comptes consolidés
  m02-f       13×  Volledig schema kapitaalvennootschap
  m02-f       13×  Schéma complet entreprise à capital
  m122-f-p     9×  Comptes consolidés d'un consortium
  m122-f-p     9×  Geconsolideerde jaarrekening betreffende een consortium
  m02-f-p      8×  PDF - Volledig schema kapitaalvennootschap
  m02-f-p      8×  PDF - Schéma complet entreprise à capital
  m02-f        5×  Modèle complet société à capital
  m02-f        5×  Volledig model kapitaalvennootschap

Après filtres (comptes_retenus) : 0 année(s)

Exemple de dépôt :
{
  "periodEndDateYear": 2025,
  "modelId": "m120-f-p",
  "modelName": "Modèle pour comptes consolidés",
  "language": "FR",
  "reference": "2026-00162246",
  "id": "b69da26d-63cf-11f1-9e09-db0606b28a82"
}


---
### 2.8) Établissements

Liste de chaque établissement de l'entreprise.

| Numéro | Adresse | Date de création | Activité |
|---|---|---|---|

In [85]:
KBO = "https://kbopub.economie.fgov.be/kbopub"

def _kbo_get(path, params):
    r = requests.get(f"{KBO}/{path}", params=params, headers=HEADERS, timeout=20)
    r.encoding = "utf-8"
    return r                  

def _num9(num_url):
    return re.sub(r"\D", "", num_url).lstrip("0")

def _clean_adresse(txt):
    txt = re.split(r"\b(?:Sinds|Depuis)\b", txt)[0]
    return re.sub(r"\s+", " ", txt).strip().rstrip(",")

def etablissements_liste(num_url, lang="fr", max_pages=100, debug=False):
    numero, etabs, page = _num9(num_url), [], 1
    while page <= max_pages:
        r = _kbo_get("vestiginglijst.html",
                     {"lang": lang, "ondernemingsnummer": numero, "page": page})
        if r.status_code == 404:         
            break
        r.raise_for_status()
        soup  = BeautifulSoup(r.text, "html.parser")
        liens = soup.find_all("a", href=re.compile(r"toonvestigingps\.html\?vestigingsnummer="))
        if not liens:
            break
        for a in liens:
            tds = (a.find_parent("tr").find_all("td")) if a.find_parent("tr") else []
            etabs.append({
                "numero":        re.sub(r"\D", "", a.get_text()),
                "date_creation": tds[3].get_text(" ", strip=True) if len(tds) > 3 else "",
                "adresse":       _clean_adresse(tds[5].get_text(" ", strip=True)) if len(tds) > 5 else "",
            })
        if debug:
            print(f"  page {page}: {len(liens)} établissements (cumul {len(etabs)})")

        m = re.search(r"(\d+)\s+(?:vestigingseenheden|unité|unités|établissement)", r.text, re.I)
        total = int(m.group(1)) if m else None
        if total is not None and len(etabs) >= total:
            break
        page += 1
        time.sleep(0.25)
    return etabs

In [86]:
def _activite_etab(numero, lang="fr"):
    r = _kbo_get("toonvestigingps.html", {"lang": lang, "vestigingsnummer": numero})
    if r.status_code != 200:
        return ""
    soup = BeautifulSoup(r.text, "html.parser")        
    flat = re.sub(r"\s+", " ", soup.get_text(" "))
    m = re.search(r"(\d{2}\.\d{3})\s*-\s*(.+?)\s+(?:Depuis|Sinds)\b", flat)
    return f"{m.group(1).replace('.', '')} - {m.group(2).strip()}" if m else ""

def etablissements_kbo(num_url, with_activite=True, max_etabs=None, lang="fr", debug=False):
    etabs = etablissements_liste(num_url, lang=lang, debug=debug)
    if max_etabs:
        etabs = etabs[:max_etabs]
    for i, e in enumerate(etabs, 1):
        e["activite"] = _activite_etab(e["numero"], lang=lang) if with_activite else ""
        if debug and with_activite and i % 20 == 0:
            print(f"  activités récupérées : {i}/{len(etabs)}")
        if with_activite:
            time.sleep(0.25)
    return [{"numero": e["numero"], "adresse": e["adresse"],
             "date_creation": e["date_creation"], "activite": e["activite"]} for e in etabs]

In [87]:
def afficher_etablissements(num_url, with_activite=True, max_etabs=None):
    etabs = etablissements_kbo(num_url, with_activite=with_activite,
                               max_etabs=max_etabs, debug=True)
    df = pd.DataFrame(etabs, columns=["numero", "adresse", "date_creation", "activite"])
    df.columns = ["Numéro", "Adresse", "Date de création", "Activité"]
    print("=" * 60)
    print(f"  Établissements — {num_url}  ({len(df)} établissement·s)")
    print("=" * 60)
    display(df)
    return etabs

google_etabs = afficher_etablissements(GOOGLE_NUM_URL)
apple_etabs  = afficher_etablissements(APPLE_NUM_URL)
sncb_etabs   = afficher_etablissements(SNCB_NUM_URL)

  page 1: 1 établissements (cumul 1)
  Établissements — 0878065378  (1 établissement·s)


,Numéro,Adresse,Date de création,Activité
0,2151627472,Chaussée d'Etterbeek 180 1040 Etterbeek,10 février 2006,62900 - Autres activités de service informatique


  page 1: 2 établissements (cumul 2)
  Établissements — 0836157420  (2 établissement·s)


,Numéro,Adresse,Date de création,Activité
0,2200008005,Boulevard Saint-Lazare 4-10 1210 Saint-Josse-t...,6 mai 2011,47400 - Commerce de détail d’équipements de l’...
1,2244110935,Avenue de la Toison d'Or 26-28 1050 Ixelles,19 septembre 2015,47400 - Commerce de détail d’équipements de l’...


  page 1: 20 établissements (cumul 20)
  page 2: 20 établissements (cumul 40)
  page 3: 20 établissements (cumul 60)
  page 4: 20 établissements (cumul 80)
  page 5: 20 établissements (cumul 100)
  page 6: 20 établissements (cumul 120)
  page 7: 20 établissements (cumul 140)
  page 8: 20 établissements (cumul 160)
  page 9: 20 établissements (cumul 180)
  page 10: 20 établissements (cumul 200)
  page 11: 20 établissements (cumul 220)
  page 12: 20 établissements (cumul 240)
  page 13: 20 établissements (cumul 260)
  page 14: 10 établissements (cumul 270)
  activités récupérées : 20/270
  activités récupérées : 40/270
  activités récupérées : 60/270
  activités récupérées : 80/270
  activités récupérées : 100/270
  activités récupérées : 120/270
  activités récupérées : 140/270
  activités récupérées : 160/270
  activités récupérées : 180/270
  activités récupérées : 200/270
  activités récupérées : 220/270
  activités récupérées : 240/270
  activités récupérées : 260/270
  Établissemen

,Numéro,Adresse,Date de création,Activité
0,2143271418,Cantersteen 10 1000 Bruxelles,1 janvier 1968,49110 - Transport ferroviaire lourd de voyageurs
1,2143272111,Boulevard de l'Impératrice 5 1000 Bruxelles,1 janvier 1968,49110 - Transport ferroviaire lourd de voyageurs
2,2143273594,Place Princesse Elisabeth 5 1030 Schaerbeek,1 janvier 1968,49110 - Transport ferroviaire lourd de voyageurs
3,2143273693,Place Princesse Elisabeth 6 1030 Schaerbeek,1 janvier 1968,49110 - Transport ferroviaire lourd de voyageurs
4,2143273891,Rue du Progrès 76 1030 Schaerbeek,1 janvier 1968,49110 - Transport ferroviaire lourd de voyageurs
...,...,...,...,...
265,2285820242,Rue des Deux Gares 84 1070 Anderlecht,1 janvier 2019,49110 - Transport ferroviaire lourd de voyageurs
266,2291131882,Marksesteenweg(Kor) 165 8500 Kortrijk,1 mai 2019,49110 - Transport ferroviaire lourd de voyageurs
267,2291141285,Konterdamkaai 12 8400 Oostende,1 juin 2019,49110 - Transport ferroviaire lourd de voyageurs
268,2297380365,Koningin Fabiolalaan 190 9000 Gent,1 décembre 2019,49110 - Transport ferroviaire lourd de voyageurs


---
### 2.9) Publications (eJustice)

Source : https://www.ejustice.just.fgov.be/cgi_tsv/list.pl?language=fr&btw={tva}&page=1

| Date | Référence (NUMAC) | Type | Lien |
|---|---|---|---|

In [88]:
EJ_BASE = "https://www.ejustice.just.fgov.be/cgi_tsv/list.pl"

def _btw(num_url):
    return re.sub(r"\D", "", num_url).zfill(10)      # -> 0203430576

def ejustice_page(num_url, page=1):
    params = {"language": "fr", "btw": _btw(num_url), "page": page}
    r = requests.get(EJ_BASE, params=params, headers=HEADERS, timeout=20)
    r.encoding = "utf-8"
    return r

In [89]:
r = ejustice_page(SNCB_NUM_URL, page=1)
print("status:", r.status_code, "| taille:", len(r.text))

soup = BeautifulSoup(r.text, "html.parser")

liens = soup.find_all("a", href=re.compile(r"numac|pdf|article", re.I))
print("liens publication trouvés :", len(liens))

if liens:
    bloc = liens[0].find_parent(["div", "tr", "li", "p"]) or liens[0].parent
    print("\n--- 1er href ---\n", liens[0].get("href"))
    print("\n--- bloc parent (HTML brut) ---\n", str(bloc)[:1200])

pag = soup.find_all("a", href=re.compile(r"page=\d+"))
print("\nliens de pagination :", sorted({re.search(r'page=(\d+)', a['href']).group(1) for a in pag}))

status: 200 | taille: 87002
liens publication trouvés : 297

--- 1er href ---
 article.pl?language=fr&btw_search=203430576&page=1&la_search=f&caller=list&=0&btw=203430576

--- bloc parent (HTML brut) ---
 <div class="list-item--content">
<p class="list-item--subtitle">
1) <font color="blue">SOCIETE NATIONALE DES CHEMINS DE FER BELGES, EN ABREGE : SNCB</font>  SA

</p>
<a class="list-item--title" href="article.pl?language=fr&amp;btw_search=203430576&amp;page=1&amp;la_search=f&amp;caller=list&amp;=0&amp;btw=203430576">
RUE DE FRANCE 56 1060 SAINT-GILLES
<br/>
203.430.576
<br/>
DEMISSIONS, NOMINATIONS 
<br/>
2026-06-23 / 0080130       <a class="standard" href="/tsv_pdf/2026/06/23/26080130.pdf" target="_blank">IMAGE</a>
<br/>
<br/>
</a>
</div>

liens de pagination : ['1', '2', '3', '4']


In [90]:
EJ_ROOT = "https://www.ejustice.just.fgov.be"

def publications_ejustice(num_url, max_pages=100, debug=False):
    pubs, seen, page = [], set(), 1
    while page <= max_pages:
        r = ejustice_page(num_url, page=page)
        if r.status_code != 200:
            break
        soup   = BeautifulSoup(r.text, "html.parser")
        blocks = soup.select("div.list-item--content")
        if not blocks:                 
            break
        n0 = len(pubs)
        for b in blocks:
            pdf = b.find("a", href=re.compile(r"/tsv_pdf/.+\.pdf", re.I))
            txt = b.get_text(" ", strip=True)
            m   = re.search(r"(\d{4}-\d{2}-\d{2})\s*/\s*(\w+)", txt) 
            pub = {
                "date":  m.group(1) if m else "",
                "numac": m.group(2) if m else "",
                "type":  pdf.get_text(strip=True) if pdf else "",  
                "lien":  EJ_ROOT + pdf["href"] if pdf else "",
            }
            key = (pub["numac"], pub["lien"])
            if key in seen:
                continue
            seen.add(key)
            pubs.append(pub)
        if debug:
            print(f"  page {page}: +{len(pubs)-n0} publications (cumul {len(pubs)})")
        page += 1
        time.sleep(0.3)
    return pubs

def afficher_publications(num_url):
    pubs = publications_ejustice(num_url, debug=True)
    df = pd.DataFrame(pubs, columns=["date", "numac", "type", "lien"])
    df.columns = ["Date", "Référence (NUMAC)", "Type", "Lien"]
    print("=" * 60)
    print(f"  Publications eJustice — {num_url}  ({len(df)} publication·s)")
    print("=" * 60)
    display(df)
    return pubs

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_publications(n)
    time.sleep(1)

  page 1: +52 publications (cumul 52)
  Publications eJustice — 0878065378  (52 publication·s)


,Date,Référence (NUMAC),Type,Lien
0,2025-11-19,0146796,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2025...
1,2025-09-05,0112455,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2025...
2,2025-02-21,0024685,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2025...
3,2024-06-06,0085745,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2024...
4,2023-08-01,0098698,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2023...
5,2022-07-12,0345863,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2022...
6,2022-04-26,0052617,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2022...
7,2022-02-24,0025737,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2022...
8,2021-08-19,0099839,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2021...
9,2021-02-15,0019944,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2021...


  page 1: +25 publications (cumul 25)
  Publications eJustice — 0836157420  (25 publication·s)


,Date,Référence (NUMAC),Type,Lien
0,2025-01-24,0012542,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2025...
1,2025-01-10,0005006,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2025...
2,2023-02-09,0019943,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2023...
3,2021-09-23,0355681,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2021...
4,2021-05-12,0056904,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2021...
5,2018-04-24,0066411,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2018...
6,2018-03-06,0041051,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2018...
7,2017-08-23,0121934,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2017...
8,2017-04-12,0048635,,
9,2017-02-20,0027011,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2017...


  page 1: +100 publications (cumul 100)
  page 2: +100 publications (cumul 200)
  page 3: +93 publications (cumul 293)
  page 4: +42 publications (cumul 335)
  Publications eJustice — 0203430576  (335 publication·s)


,Date,Référence (NUMAC),Type,Lien
0,2026-06-23,0080130,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2026...
1,2026-06-23,0080131,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2026...
2,2026-05-28,0068069,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2026...
3,2026-05-28,0068070,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2026...
4,2025-12-03,0152497,IMAGE,https://www.ejustice.just.fgov.be/tsv_pdf/2025...
...,...,...,...,...
330,1986-01-24,059,,
331,1985-01-08,631,,
332,1985-01-08,632,,
333,1984-01-01,0240,,


---
### 2.10) Informations de Contact

| Champ | Valeur |
|---|---|
| Téléphone | |
| Email | |
| Site web | |
| Adresse | |

In [91]:
def infos_contact(num_url):
    soup = kbo_soup(num_url)
    kv   = _kv(soup)

    def _clean(v):
        return "" if (not v or "pas de donn" in v.lower()) else v.split("\n")[0].strip()

    adr = _clean(kv.get("Adresse du siège", ""))
    return {
        "telephone": _clean(kv.get("Numéro de téléphone", "")),
        "email":     _clean(kv.get("E-mail", "")),
        "site_web":  _clean(kv.get("Adresse web", "")),
        "adresse":   re.sub(r"\s+", " ", adr).strip(),
    }

def afficher_contact(num_url):
    c = infos_contact(num_url)
    print("=" * 60)
    print(f"  Informations de contact — {num_url}")
    print("=" * 60)
    libelles = {"telephone": "Téléphone", "email": "Email",
                "site_web": "Site web", "adresse": "Adresse"}
    for k, lib in libelles.items():
        print(f"  {lib:<10}: {c[k] if c[k] else '—'}")
    print()
    return c

for n in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    afficher_contact(n)
    time.sleep(1)

  Informations de contact — 0878065378
  Téléphone : —
  Email     : —
  Site web  : —
  Adresse   : Chaussée d'Etterbeek 180

  Informations de contact — 0836157420
  Téléphone : —
  Email     : —
  Site web  : —
  Adresse   : Boulevard Saint-Lazare 4-10

  Informations de contact — 0203430576
  Téléphone : —
  Email     : —
  Site web  : —
  Adresse   : Rue de France 56



In [116]:
# ════════════════════════════════════════════════════════════════════
#  COUCHE D'INGESTION — Data Lake (raw / bronze)
#  Stocke les réponses BRUTES de chaque source, sans transformation.
#  Arborescence : data/raw/{source}/{numero}/{fichier}
# ════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

RAW = Path("data/raw")

def save_raw(source, num_url, filename, content):
    """Écrit une réponse brute dans le lake. content: str (texte) ou bytes (binaire)."""
    numero = re.sub(r"\D", "", num_url).zfill(10)
    dest = RAW / source / numero / filename
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(content) if isinstance(content, (bytes, bytearray)) else \
        dest.write_text(content, encoding="utf-8")
    return dest

def _get(url, **kw):
    """GET brut avec tes headers par défaut."""
    kw.setdefault("headers", HEADERS); kw.setdefault("timeout", 30)
    r = requests.get(url, **kw); r.encoding = "utf-8"
    return r

# ── 1. kbopub : fiche entreprise (sert pour 2.1/2.2/2.3/2.4/2.5/2.10) ──
def ingest_kbopub_fiche(num_url):
    r = _get(BASE_URL, params={"lang": "fr", "ondernemingsnummer": num_url})
    save_raw("kbopub", num_url, "entreprise.html", r.text)

# ── 2. kbopub : établissements (liste paginée + fiche de chaque UE) ──
def ingest_kbopub_etablissements(num_url, max_etabs=None):
    numero = _num9(num_url)
    page, vus = 1, []
    while True:
        r = _get(f"{KBO}/vestiginglijst.html",
                 params={"lang": "fr", "ondernemingsnummer": numero, "page": page})
        if r.status_code == 404:
            break
        save_raw("kbopub", num_url, f"etablissements_p{page}.html", r.text)
        soup  = BeautifulSoup(r.text, "html.parser")
        liens = soup.find_all("a", href=re.compile(r"toonvestigingps\.html\?vestigingsnummer="))
        if not liens:
            break
        for a in liens:
            vus.append(re.sub(r"\D", "", a.get_text()))
        m = re.search(r"(\d+)\s+(?:vestigingseenheden|unité|unités|établissement)", r.text, re.I)
        total = int(m.group(1)) if m else None
        if total is not None and len(vus) >= total:
            break
        page += 1; time.sleep(0.25)
    if max_etabs:
        vus = vus[:max_etabs]
    for vest in vus:                       # fiche brute de chaque établissement
        rv = _get(f"{KBO}/toonvestigingps.html", params={"lang": "fr", "vestigingsnummer": vest})
        if rv.status_code == 200:
            save_raw("kbopub", num_url, f"vestiging_{vest}.html", rv.text)
        time.sleep(0.25)

# ── 3. eJustice : pages de publications (HTML paginé) ──
def ingest_ejustice(num_url):
    page = 1
    while page <= 100:
        r = _get(EJ_BASE, params={"language": "fr", "btw": _btw(num_url), "page": page})
        if r.status_code != 200:
            break
        soup = BeautifulSoup(r.text, "html.parser")
        if not soup.select("div.list-item--content"):
            break
        save_raw("ejustice", num_url, f"page_{page}.html", r.text)
        page += 1; time.sleep(0.3)

# ── 4. notaire.be : JSON brut des statuts ──
def ingest_notaire(num_url, cookie=None):
    cookie = cookie or get_cookie_notaire()
    items  = statuts_kbo(num_url, cookie=cookie)
    if not items and not _cookie_valide(cookie):   # cookie mort en route -> régénère
        cookie = get_cookie_notaire(force=True)
        items  = statuts_kbo(num_url, cookie=cookie)
    save_raw("notaire", num_url, "statutes.json",
             json.dumps(items, ensure_ascii=False, indent=2))
    return cookie

# ── 5. CBSO/NBB : JSON brut des dépôts + binaires PDF/CSV/XBRL ──
def ingest_cbso(num_url):
    numero = re.sub(r"\D", "", num_url).zfill(10)
    deps = cbso_deposits(numero)
    save_raw("cbso", num_url, "deposits.json",
             json.dumps(deps, ensure_ascii=False, indent=2))

    # en-têtes pour FICHIERS binaires : accepter tout (pas application/json !)
    h_bin = {"User-Agent": "Mozilla/5.0", "Accept": "*/*",
             "Referer": "https://consult.cbso.nbb.be/"}

    for annee, dep in comptes_retenus(numero).items():
        did = dep["id"]
        rp = _get(f"{BROKER}/pdf/{did}", headers=h_bin)
        if rp.status_code == 200 and rp.content:
            save_raw("cbso", num_url, f"pdf/{annee}.pdf", rp.content)
        rx = _get(f"{BROKER}/xbrl/{did}", headers=h_bin)
        if rx.status_code == 200 and rx.content:
            save_raw("cbso", num_url, f"xbrl/{annee}.xbrl", rx.content)
        if annee >= 2021:
            rc = _get(f"{BROKER}/consult/csv/{did}", headers=h_bin)
            if rc.status_code == 200 and rc.content:
                save_raw("cbso", num_url, f"csv/{annee}.csv", rc.content)
        time.sleep(0.4)

# ── Orchestrateur : ingère tout, pour les 3 entreprises ──
def ingest_all(cookie_notaire=None, max_etabs=None):
    for num in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
        print(f"\n=== Ingestion {num} ===")
        ingest_kbopub_fiche(num);                 print("  kbopub fiche ✓")
        ingest_kbopub_etablissements(num, max_etabs=max_etabs); print("  kbopub établissements ✓")
        ingest_ejustice(num);                     print("  eJustice ✓")
        ingest_notaire(num, cookie=cookie_notaire); print("  notaire ✓")
        ingest_cbso(num);                         print("  cbso ✓")
    print("\nLake raw écrit dans :", RAW.resolve())

In [117]:
# régénère un cookie frais (cache si encore valide, sinon Chrome 3s)
COOKIE_NOTAIRE = get_cookie_notaire()

# SNCB a 270 établissements -> max_etabs=20 pour un lake raisonnable en démo
ingest_all(cookie_notaire=COOKIE_NOTAIRE, max_etabs=20)

Cookie notaire OK (cache)

=== Ingestion 0878065378 ===
  kbopub fiche ✓
  kbopub établissements ✓
  eJustice ✓
  notaire ✓
  cbso ✓

=== Ingestion 0836157420 ===
  kbopub fiche ✓
  kbopub établissements ✓
  eJustice ✓
  notaire ✓
  cbso ✓

=== Ingestion 0203430576 ===
  kbopub fiche ✓
  kbopub établissements ✓
  eJustice ✓
Cookie renouvelé → notaire_cookies.json
  notaire ✓
  cbso ✓

Lake raw écrit dans : C:\Users\FA706\IpssiDJ\bigdata-finance\data\raw


In [105]:
# ── Vérification de la couche raw (à lancer après ingest_all) ──
from pathlib import Path

RAW = Path("data/raw")

def verifier_lake():
    sources_attendues = ["kbopub", "ejustice", "notaire", "cbso"]
    for num in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
        numero = re.sub(r"\D", "", num).zfill(10)
        print("=" * 60)
        print(f"  {numero}")
        print("=" * 60)
        for src in sources_attendues:
            base = RAW / src / numero
            if not base.exists():
                print(f"  {src:9s} : ❌ AUCUN fichier")
                continue
            fichiers = sorted(p.relative_to(base).as_posix() for p in base.rglob("*") if p.is_file())
            taille   = sum(p.stat().st_size for p in base.rglob("*") if p.is_file())
            print(f"  {src:9s} : {len(fichiers):3d} fichier·s  ({taille/1024:.0f} Ko)")
            # détail compact
            apercu = fichiers[:6] + (["…"] if len(fichiers) > 6 else [])
            for f in apercu:
                print(f"            └ {f}")
        print()

verifier_lake()

  0878065378
  kbopub    :   3 fichier·s  (41 Ko)
            └ entreprise.html
            └ etablissements_p1.html
            └ vestiging_2151627472.html
  ejustice  :   1 fichier·s  (46 Ko)
            └ page_1.html
  notaire   :   1 fichier·s  (0 Ko)
            └ statutes.json
  cbso      :  16 fichier·s  (6968 Ko)
            └ csv/2021.csv
            └ csv/2022.csv
            └ csv/2023.csv
            └ csv/2024.csv
            └ csv/2025.csv
            └ deposits.json
            └ …

  0836157420
  kbopub    :   4 fichier·s  (47 Ko)
            └ entreprise.html
            └ etablissements_p1.html
            └ vestiging_2200008005.html
            └ vestiging_2244110935.html
  ejustice  :   1 fichier·s  (26 Ko)
            └ page_1.html
  notaire   :   1 fichier·s  (0 Ko)
            └ statutes.json
  cbso      :  15 fichier·s  (4651 Ko)
            └ csv/2022.csv
            └ csv/2023.csv
            └ csv/2024.csv
            └ csv/2025.csv
            └ deposits.jso

In [110]:
COOKIE_NOTAIRE = get_cookie_notaire()        # cookie frais maintenant
for num in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    ingest_notaire(num, cookie=COOKIE_NOTAIRE)

# vérif par contenu
for num in [GOOGLE_NUM_URL, APPLE_NUM_URL, SNCB_NUM_URL]:
    p = RAW/"notaire"/re.sub(r"\D","",num).zfill(10)/"statutes.json"
    print(num, "->", len(json.loads(p.read_text())), "statut(s)")

Cookie notaire OK (cache)
0878065378 -> 1 statut(s)
0836157420 -> 1 statut(s)
0203430576 -> 0 statut(s)


---
## 3) Finances

Tableau financier depuis 2021 (ou depuis la création) jusqu'à 2025, enrichi avec l'EBIT.

### 📊 Tableau des indicateurs financiers

#### Performance
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Chiffre d'affaires | | | | | |
| Marge brute | | | | | |
| EBIT (Résultat d'exploitation) | | | | | |
| Résultat net | | | | | |

#### Croissance
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Taux de croissance du CA (%) | | | | | |
| Taux de marge brute (%) | | | | | |
| % de marge nette | | | | | |

#### Autonomie Financière
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Trésorerie | | | | | |
| Dettes financières | | | | | |
| Dette financière nette | | | | | |

#### Solvabilité
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Fonds propres | | | | | |

#### Ressources Humaines & Coûts
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Nombre d'employés | | | | | |
| Coûts salariaux (salaires + charges + avantages) | | | | | |
| Employés en % du coût des revenus | | | | | |
| Revenu par employé | | | | | |
| Taxes payées | | | | | |

In [96]:
CSV_DIR = Path("data/nbb/csvs")
ANNEES  = [2021, 2022, 2023, 2024, 2025]

def lire_csv_nbb(path):
    df = pd.read_csv(path, sep=",", header=None, names=["k", "v"], dtype=str, quotechar='"')
    rub = {}
    for k, v in zip(df["k"], df["v"]):
        if k and re.fullmatch(r"\d+P?", str(k)):
            try: rub[k] = float(str(v).replace(",", "."))
            except (ValueError, AttributeError): pass
    return rub

def _codes(numero, annee):
    p = CSV_DIR / numero / f"{annee}.csv"
    return lire_csv_nbb(p) if p.exists() else None

def _v(r, *codes):                  
    for c in codes:
        if c in r: return r[c]
    return None

def _somme(r, codes):              
    vals = [r[c] for c in codes if c in r]
    return sum(vals) if vals else None

EQUITY  = ['10','11','12','13','14','15']
CASH    = ['50','51','52','53','54','55','56','57','58']
FINDEBT = ['170','171','172','173','174','42','43']
OPCH    = ['60','61','62','630','631','632','633','634','635','640','641','642','643','644','648','649']

def indicateurs(r):
    if r is None: return None
    ca = _v(r,'70')

    mb = _v(r,'9900')
    if mb is None and ca is not None and ('60' in r or '61' in r):
        mb = (ca + (_v(r,'74') or 0) - (_v(r,'60') or 0) - (_v(r,'61') or 0))

    cash, fdebt = _somme(r, CASH), _somme(r, FINDEBT)
    return {
        "ca": ca,
        "marge_brute": mb,
        "ebit": _v(r,'9901','9900'),            
        "net":  _v(r,'9904','9905'),
        "tresorerie": cash,
        "dettes_fin": fdebt,
        "dette_nette": (fdebt - (cash or 0)) if fdebt is not None else None,
        "fonds_propres": _somme(r, EQUITY),
        "effectif": _v(r,'9087','9086','9097','1003'),       
        "remun":    _v(r,'62','620'),   
        "opch":     _somme(r, OPCH) or _v(r,'9125'),        
        "taxes":    _v(r,'9903'),
    }

In [97]:
def tableau_finance(numero):
    numero = numero.replace(".", "").zfill(10)
    ind = {a: indicateurs(_codes(numero, a)) for a in ANNEES}
    g   = lambda a, k: (ind[a].get(k) if ind[a] else None)

    def ratio(num_k, den_k):
        return {a: (g(a,num_k)/g(a,den_k)*100) if (g(a,num_k) is not None and g(a,den_k)) else None for a in ANNEES}

    rows, crois = {}, {}
    for i, a in enumerate(ANNEES):
        ca, prev = g(a,"ca"), (g(ANNEES[i-1],"ca") if i>0 else None)
        crois[a] = (ca/prev-1)*100 if (ca and prev) else None

    rows[("Performance","Chiffre d'affaires")]              = {a:g(a,"ca")  for a in ANNEES}
    rows[("Performance","Marge brute")]                     = {a:g(a,"marge_brute") for a in ANNEES}
    rows[("Performance","EBIT (Résultat d'exploitation)")]  = {a:g(a,"ebit") for a in ANNEES}
    rows[("Performance","Résultat net")]                    = {a:g(a,"net")  for a in ANNEES}
    rows[("Croissance","Taux de croissance du CA (%)")]     = crois
    rows[("Croissance","Taux de marge brute (%)")]          = ratio("marge_brute","ca")
    rows[("Croissance","% de marge nette")]                 = ratio("net","ca")
    rows[("Autonomie financière","Trésorerie")]             = {a:g(a,"tresorerie") for a in ANNEES}
    rows[("Autonomie financière","Dettes financières")]     = {a:g(a,"dettes_fin") for a in ANNEES}
    rows[("Autonomie financière","Dette financière nette")] = {a:g(a,"dette_nette") for a in ANNEES}
    rows[("Solvabilité","Fonds propres")]                   = {a:g(a,"fonds_propres") for a in ANNEES}
    rows[("RH & Coûts","Nombre d'employés")]                = {a:g(a,"effectif") for a in ANNEES}
    rows[("RH & Coûts","Coûts salariaux (salaires + charges + avantages)")] = {a:g(a,"remun") for a in ANNEES}
    rows[("RH & Coûts","Employés en % du coût des revenus")] = ratio("remun","opch")
    rows[("RH & Coûts","Revenu par employé")]               = {a:(g(a,"ca")/g(a,"effectif")) if (g(a,"ca") and g(a,"effectif")) else None for a in ANNEES}
    rows[("RH & Coûts","Taxes payées")]                     = {a:g(a,"taxes") for a in ANNEES}

    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index = pd.MultiIndex.from_tuples(df.index, names=["Catégorie","Indicateur"])
    df.columns = ANNEES
    return df

def _fmt(df):
    out = df.copy().astype(object)
    for (cat, ind_), row in df.iterrows():
        pct = "%" in ind_
        for a in df.columns:
            v = row[a]
            out.loc[(cat, ind_), a] = "—" if (v is None or pd.isna(v)) else (
                f"{v:,.1f}%".replace(",", " ") if pct else f"{v:,.0f}".replace(",", " "))
    return out

for num, nom in [(GOOGLE_NUM_URL,"GOOGLE BELGIUM"),
                 (APPLE_NUM_URL,"APPLE RETAIL BELGIUM"),
                 (SNCB_NUM_URL,"SNCB")]:
    print("="*72); print("  ", nom, f"({num})"); print("="*72)
    display(_fmt(tableau_finance(num)))

   GOOGLE BELGIUM (0878065378)


2021  \
Catégorie            Indicateur                                                     
Performance          Chiffre d'affaires                                63 066 577   
                     Marge brute                                       47 736 183   
                     EBIT (Résultat d'exploitation)                     8 711 228   
                     Résultat net                                       6 529 995   
Croissance           Taux de croissance du CA (%)                               —   
                     Taux de marge brute (%)                                75.7%   
                     % de marge nette                                       10.4%   
Autonomie financière Trésorerie                                                 —   
                     Dettes financières                                         —   
                     Dette financière nette                                     —   
Solvabilité          Fonds propres                                     30 438 332   
RH & Coûts           Nombre d'employés                                        105   
                     Coûts salariaux (salaires + charges + avantages)  37 434 138   
                     Employés en % du coût des revenus                      68.9%   
                     Revenu par employé                                   598 355   
                     Taxes payées                                       8 653 215   

                                                                             2022  \
Catégorie            Indicateur                                                     
Performance          Chiffre d'affaires                                77 386 505   
                     Marge brute                                       60 131 435   
                     EBIT (Résultat d'exploitation)                    12 796 038   
                     Résultat net                                       9 532 020   
Croissance           Taux de croissance du CA (%)                           22.7%   
                     Taux de marge brute (%)                                77.7%   
                     % de marge nette                                       12.3%   
Autonomie financière Trésorerie                                                 —   
                     Dettes financières                                         —   
                     Dette financière nette                                     —   
Solvabilité          Fonds propres                                     39 970 352   
RH & Coûts           Nombre d'employés                                        130   
                     Coûts salariaux (salaires + charges + avantages)  45 517 134   
                     Employés en % du coût des revenus                      70.5%   
                     Revenu par employé                                   595 281   
                     Taxes payées                                      12 888 842   

                                                                             2023  \
Catégorie            Indicateur                                                     
Performance          Chiffre d'affaires                                94 095 721   
                     Marge brute                                       75 975 463   
                     EBIT (Résultat d'exploitation)                    15 185 005   
                     Résultat net                                      12 471 274   
Croissance           Taux de croissance du CA (%)                           21.6%   
                     Taux de marge brute (%)                                80.7%   
                     % de marge nette                                       13.3%   
Autonomie financière Trésorerie                                                 —   
                     Dettes financières                                         —   
                     Dette financière nette                                     

   APPLE RETAIL BELGIUM (0836157420)


2021  \
Catégorie            Indicateur                                              
Performance          Chiffre d'affaires                                  —   
                     Marge brute                                         —   
                     EBIT (Résultat d'exploitation)                      —   
                     Résultat net                                        —   
Croissance           Taux de croissance du CA (%)                        —   
                     Taux de marge brute (%)                             —   
                     % de marge nette                                    —   
Autonomie financière Trésorerie                                          —   
                     Dettes financières                                  —   
                     Dette financière nette                              —   
Solvabilité          Fonds propres                                       —   
RH & Coûts           Nombre d'employés                                   —   
                     Coûts salariaux (salaires + charges + avantages)    —   
                     Employés en % du coût des revenus                   —   
                     Revenu par employé                                  —   
                     Taxes payées                                        —   

                                                                             2022  \
Catégorie            Indicateur                                                     
Performance          Chiffre d'affaires                                65 967 024   
                     Marge brute                                       11 308 747   
                     EBIT (Résultat d'exploitation)                     1 703 990   
                     Résultat net                                       1 094 546   
Croissance           Taux de croissance du CA (%)                               —   
                     Taux de marge brute (%)                                17.1%   
                     % de marge nette                                        1.7%   
Autonomie financière Trésorerie                                                 —   
                     Dettes financières                                         —   
                     Dette financière nette                                     —   
Solvabilité          Fonds propres                                      4 506 432   
RH & Coûts           Nombre d'employés                                        122   
                     Coûts salariaux (salaires + charges + avantages)   7 858 522   
                     Employés en % du coût des revenus                      12.3%   
                     Revenu par employé                                   540 270   
                     Taxes payées                                       1 554 208   

                                                                             2023  \
Catégorie            Indicateur                                                     
Performance          Chiffre d'affaires                                64 378 152   
                     Marge brute                                       11 802 420   
                     EBIT (Résultat d'exploitation)                     1 801 111   
                     Résultat net                                       1 426 626   
Croissance           Taux de croissance du CA (%)                           -2.4%   
                     Taux de marge brute (%)                                18.3%   
                     % de marge nette                                        2.2%   
Autonomie financière Trésorerie                                                 —   
                     Dettes financières                                         —   
                     Dette financière nette                                     —   
Solvabilité          Fonds propres                                      5 933 058   
RH & Coûts           Nombre d

   SNCB (0203430576)


2021  \
Catégorie            Indicateur                                                        
Performance          Chiffre d'affaires                                1 941 915 999   
                     Marge brute                                        -207 199 380   
                     EBIT (Résultat d'exploitation)                     -406 155 484   
                     Résultat net                                           -328 878   
Croissance           Taux de croissance du CA (%)                                  —   
                     Taux de marge brute (%)                                  -10.7%   
                     % de marge nette                                          -0.0%   
Autonomie financière Trésorerie                                          245 339 865   
                     Dettes financières                                2 886 775 343   
                     Dette financière nette                            2 641 435 478   
Solvabilité          Fonds propres                                     7 357 397 231   
RH & Coûts           Nombre d'employés                                        17 147   
                     Coûts salariaux (salaires + charges + avantages)              —   
                     Employés en % du coût des revenus                             —   
                     Revenu par employé                                      113 249   
                     Taxes payées                                           -187 872   

                                                                                2022  \
Catégorie            Indicateur                                                        
Performance          Chiffre d'affaires                                2 215 187 323   
                     Marge brute                                         -91 206 191   
                     EBIT (Résultat d'exploitation)                     -438 302 424   
                     Résultat net                                         17 484 275   
Croissance           Taux de croissance du CA (%)                              14.1%   
                     Taux de marge brute (%)                                   -4.1%   
                     % de marge nette                                           0.8%   
Autonomie financière Trésorerie                                          192 022 084   
                     Dettes financières                                2 831 393 578   
                     Dette financière nette                            2 639 371 494   
Solvabilité          Fonds propres                                     7 555 339 478   
RH & Coûts           Nombre d'employés                                        16 771   
                     Coûts salariaux (salaires + charges + avantages)              —   
                     Employés en % du coût des revenus                             —   
                     Revenu par employé                                      132 081   
                     Taxes payées                                         17 555 453   

                                                                                2023  \
Catégorie            Indicateur                                                        
Performance          Chiffre d'affaires                                2 563 319 883   
                     Marge brute                                          66 484 577   
                     EBIT (Résultat d'exploitation)                     -390 869 200   
                     Résultat net                                         67 145 919   
Croissance           Taux de croissance du CA (%)                              15.7%   
                     Taux de marge brute (%)                                    2.6%   
                     % de marge nette                                           2.6%   
Autonomie financière Trésorerie                                          244 608 393   
                     Dettes fi

In [98]:
def sankey_resultat(numero, nom, annee=2024):
    numero = numero.replace(".", "").zfill(10)
    r = _codes(numero, annee)
    if r is None:
        print(f"{nom} : pas de CSV {annee}"); return
    ca    = _v(r,'70') or 0
    ebit  = _v(r,'9901','9900') or 0
    net   = _v(r,'9904','9905') or 0
    taxes = _v(r,'9903') or 0
    cout_ventes = max(ca - ebit, 0)       
    rai   = net + taxes                   
    fin   = rai - ebit                        

    labels = ["Chiffre d'affaires", "Charges d'exploitation", "EBIT",
              "Résultat avant impôts", "Impôts", "Résultat net", "Apport financier"]
    CA, CHG, EBIT, RAI, TAX, NET, FIN = range(7)

    src, tgt, val, col = [], [], [], []
    def flux(s, t, v, c):
        if v and v > 0:
            src.append(s); tgt.append(t); val.append(v); col.append(c)

    flux(CA,  CHG,  cout_ventes, "rgba(214,39,40,0.45)")    
    flux(CA,  EBIT, max(ebit,0), "rgba(44,160,44,0.45)")     
    if fin >= 0:
        flux(EBIT, RAI, max(ebit,0), "rgba(44,160,44,0.45)")
        flux(FIN,  RAI, fin,        "rgba(31,119,180,0.45)")
    else:
        flux(EBIT, RAI, max(rai,0), "rgba(44,160,44,0.45)")  
    flux(RAI, TAX, max(taxes,0), "rgba(214,39,40,0.45)")     
    flux(RAI, NET, max(net,0),   "rgba(44,160,44,0.55)") 

    fig = go.Figure(go.Sankey(
        node=dict(label=labels, pad=18, thickness=18,
                  color=["#1f77b4","#d62728","#2ca02c","#9467bd","#d62728","#2ca02c","#1f77b4"]),
        link=dict(source=src, target=tgt, value=val, color=col),
    ))
    fig.update_layout(title_text=f"{nom} — Compte de résultats {annee}",
                      font_size=12, height=420)
    fig.show()

for num, nom in [(GOOGLE_NUM_URL,"GOOGLE BELGIUM"),
                 (APPLE_NUM_URL,"APPLE RETAIL BELGIUM"),
                 (SNCB_NUM_URL,"SNCB")]:
    sankey_resultat(num, nom, annee=2024)

---
### 📈 Graphiques — Chiffre d'affaires & Résultat net (2021–2025)

In [99]:
def graphe_ca_net(numero, nom):
    numero = numero.replace(".", "").zfill(10)
    ind = {a: indicateurs(_codes(numero, a)) for a in ANNEES}
    ca  = [ (ind[a]["ca"]  if ind[a] else None) for a in ANNEES ]
    net = [ (ind[a]["net"] if ind[a] else None) for a in ANNEES ]

    fig = go.Figure()
    fig.add_bar(x=ANNEES, y=ca,  name="Chiffre d'affaires", marker_color="#1f77b4")
    fig.add_bar(x=ANNEES, y=net, name="Résultat net",       marker_color="#2ca02c")
    fig.update_layout(
        title_text=f"{nom} — Chiffre d'affaires & Résultat net (2021–2025)",
        barmode="group", height=420, xaxis=dict(tickmode="array", tickvals=ANNEES),
        yaxis_title="EUR", legend=dict(orientation="h", y=1.1),
    )
    fig.show()

for num, nom in [(GOOGLE_NUM_URL,"GOOGLE BELGIUM"),
                 (APPLE_NUM_URL,"APPLE RETAIL BELGIUM"),
                 (SNCB_NUM_URL,"SNCB")]:
    graphe_ca_net(num, nom)

In [100]:
def graphe_rentabilite(numero, nom):
    numero = numero.replace(".", "").zfill(10)
    ind = {a: indicateurs(_codes(numero, a)) for a in ANNEES}
    ca     = [ (ind[a]["ca"]  if ind[a] else None) for a in ANNEES ]
    net    = [ (ind[a]["net"] if ind[a] else None) for a in ANNEES ]
    marge  = [ (n/c*100 if (n is not None and c) else None) for n, c in zip(net, ca) ]

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=ANNEES, y=ca, name="Chiffre d'affaires",
                marker_color="#3b6ea5", opacity=0.85, secondary_y=False)
    fig.add_scatter(x=ANNEES, y=marge, name="Marge nette (%)", mode="lines+markers+text",
                    text=[f"{m:.1f}%" if m is not None else "" for m in marge],
                    textposition="top center", line=dict(color="#e4572e", width=3),
                    marker=dict(size=9), secondary_y=True)

    fig.update_layout(title_text=f"{nom} — Chiffre d'affaires & marge nette (2021–2025)",
                      height=440, legend=dict(orientation="h", y=1.12),
                      xaxis=dict(tickmode="array", tickvals=ANNEES))
    fig.update_yaxes(title_text="Chiffre d'affaires (EUR)", secondary_y=False)
    fig.update_yaxes(title_text="Marge nette (%)", secondary_y=True, ticksuffix="%")
    fig.show()

for num, nom in [(GOOGLE_NUM_URL,"GOOGLE BELGIUM"),
                 (APPLE_NUM_URL,"APPLE RETAIL BELGIUM"),
                 (SNCB_NUM_URL,"SNCB")]:
    graphe_rentabilite(num, nom)

---
## 📖 Glossaire des indicateurs financiers

Tous les indicateurs ci-dessous sont calculés à partir des **codes comptables NBB** (Plan Comptable Minimum Normalisé).
Les codes entre parenthèses correspondent aux lignes du CSV téléchargé depuis la NBB.

---

### Performance

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Chiffre d'affaires (CA)** | Revenu total généré par la vente de biens et services sur l'exercice | — | `70` |
| **Marge brute** | Ce qu'il reste après déduction du coût direct des marchandises vendues | CA − Coût des marchandises | `70 − 60` |
| **EBIT** *(Résultat d'exploitation)* | Bénéfice avant intérêts et impôts — mesure la performance opérationnelle pure | — | `9901` |
| **EBITDA** | EBIT avant amortissements — proxy du cash généré par l'exploitation | EBIT + Amortissements | `9901 + 630` |
| **Résultat net** | Bénéfice final après toutes les charges, intérêts et impôts | — | `9904` |

---

### Croissance & Marges

| Indicateur | Définition | Formule |
|---|---|---|
| **Taux de croissance du CA (%)** | Variation du chiffre d'affaires d'une année sur l'autre | `(CA_n − CA_{n-1}) / CA_{n-1} × 100` |
| **Taux de marge brute (%)** | Part de la marge brute dans le CA — reflète la rentabilité commerciale | `Marge brute / CA × 100` |
| **Marge nette (%)** | Part du résultat net dans le CA — résume la rentabilité globale | `Résultat net / CA × 100` |
| **Marge EBITDA (%)** | Capacité à générer du cash avant investissements et fiscalité | `EBITDA / CA × 100` |

---

### Autonomie Financière

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Trésorerie** | Liquidités disponibles immédiatement (banque + caisse) | — | `54/58` |
| **Dettes financières** | Total des emprunts bancaires (long terme + court terme) | Dettes LT + Dettes CT | `17 + 43` |
| **Dette financière nette** | Endettement réel après déduction de la trésorerie disponible. Négatif = position de trésorerie nette (bonne santé) | Dettes financières − Trésorerie | `(17 + 43) − 54/58` |

---

### Solvabilité

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Fonds propres** | Capital apporté par les actionnaires + bénéfices accumulés non distribués | — | `10/15` |
| **Total actif** | Ensemble des ressources économiques contrôlées par l'entreprise | — | `20/58` |
| **Autonomie financière (%)** | Part des actifs financée par les fonds propres (sans recours à la dette). Plus c'est élevé, plus l'entreprise est indépendante | `Fonds propres / Total actif × 100` |

---

### Ressources Humaines

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Effectif FTE moyen** | Nombre moyen d'équivalents temps plein sur l'exercice | — | `9087` |
| **Coûts salariaux** | Rémunérations brutes + cotisations patronales + avantages extra-légaux | — | `62` |
| **Revenu par ETP** | CA généré en moyenne par chaque employé — mesure la productivité | `CA / Effectif FTE` |
| **Coût moyen par ETP** | Coût salarial total rapporté à un équivalent temps plein | `Coûts salariaux / Effectif FTE` |
| **Part salariale dans le CA (%)** | Poids des charges de personnel dans le chiffre d'affaires | `Coûts salariaux / CA × 100` |
| **Taxes payées** | Impôt des sociétés (ISOC) effectivement payé sur l'exercice | — | `670/3` |